In [7]:
import pandas as pd
import numpy as np

## **Yu+ 2018**

### Data Description

Table 1:

```
   Bytes Format Units      Label  Explanations

   2-  9 I8     ---        KIC    [757137/12934574] Kepler Input Catalog ID
  11- 16 F6.3   mag        Kpmag  [6/17] Kepler apparent magnitude
  18- 19 I2     ---        QLen   [1/19] Dataset length in quarters (1)
  21- 27 F7.2   d          DLen   [32.5/1319.1] Dataset length in days (1)
  29- 34 F6.2   uHz        numax  [3.9/273.2] Frequency of maximum
                                    oscillation power
  36- 40 F5.2   uHz      e_numax  [0.04/70.3] Uncertainty in numax
  42- 47 F6.3   uHz        Delnu  [0.7/19.3] Mean frequency separation
                                    of acoustic modes
  49- 53 F5.3   uHz      e_Delnu  [0.008/1.4] Uncertainty in Delnu
  55- 59 F5.1   ppm        A      [10.7/745.2]? Oscillation amplitude
                                    per radial mode
  61- 65 F5.1   ppm      e_A      [0.7/129]? Uncertainty in A
  67- 71 F5.1   uHz        Width  [1.2/102]? Power excess width
  73- 76 F4.1   uHz      e_Width  [0.2/47]? Uncertainty in Width
  78- 86 F9.1   ppm2/uHz   Gran   [2.7/1081447]? Granulation power at numax
  88- 95 F8.1   ppm2/uHz e_Gran   [1/142416]? Uncertainty in Gran
```

Table 2:

```
   Bytes Format Units     Label    Explanations
   1-  8 I8     ---       KIC      [757137/12934574] Kepler Input Catalog ID
  10- 13 I4     K         Teff     [3225/6356] Effective temperature (2)
  15- 17 I3     K       e_Teff     [50/682] Uncertainty in Teff
  19- 23 F5.3   [cm/s2]   log(g)   [1.4/3.4] Log surface gravity from this work
  25- 29 F5.3   [cm/s2] e_log(g)   [0.006/0.2] Uncertainty in log(g)
  31- 35 F5.2   [Sun]     [Fe/H]   [-2.5/1] Metallicity (2)
  37- 40 F4.2   [Sun]   e_[Fe/H]   [0.1/1.6] Uncertainty in [Fe/H]
  42- 45 F4.2   Msun      NoCorM   [0.4/4.6] No {Delta}{nu} correction
                                    mass solution
  47- 50 F4.2   Msun    e_NoCorM   [0.04/3] Uncertainty in NoCorM
  52- 56 F5.2   Rsun      NoCorR   [3.5/48.4] No {Delta}{nu} correction
                                    radius solution
  58- 61 F4.2   Rsun    e_NoCorR   [0.06/10] Uncertainty in NoCorR
  63- 66 F4.2   Msun      RGBcorM  [0.3/5.2] RGB {Delta}{nu} corrected
                                    mass solution
  68- 71 F4.2   Msun    e_RGBcorM  [0.03/3] Uncertainty in RGBcorM
  73- 77 F5.2   Rsun      RGBcorR  [3.2/48.7] RGB {Delta}{nu} corrected
                                    radius solution
  79- 82 F4.2   Rsun    e_RGBcorR  [0.06/10] Uncertainty in RGBcorR
  84- 87 F4.2   Msun      ClcorM   [0.4/5.3] Clump {Delta}{nu} corrected
                                    mass solution
  89- 92 F4.2   Msun    e_ClcorM   [0.04/3] Uncertainty in ClcorM
  94- 98 F5.2   Rsun      ClcorR   [3.4/51.2] Clump {Delta}{nu} corrected
                                    radius solution
 100-104 F5.2   Rsun    e_ClcorR   [0.06/10.2] Uncertainty in ClcorR
     106 I1     ---       Phase    [0/2]? Evolutionary phase (3)
```

### CDS X-Matching

Read-in xmatch.

In [8]:
array_full = pd.read_csv("../0_inputdata_CDSxmatch/Yu2018_cdsxmatch.csv")

Remove flagged entries.

In [9]:
# no flags??
print(f"Data length: {len(array_full):.0f}")

Data length: 15929


### Cohesify Data

Convert EV state, with:

 - 0 ==> not specified
 - 1 ==> likely RGB
 - 2 ==> likely CHeB

In [10]:
array_full["Phase"] = array_full["Phase"].replace({
    0: int(0),
    1: int(1),
    2: int(2)
})

Add respective uncertainties from `Yu2018_T1.dat`.

In [11]:
y18_columnnames = [
    "KIC", "Kpmag", "QLen", "DLen", "numax", "e_numax",
    "Delnu", "e_Delnu", "A", "e_A", "Width", "e_Width",
    "Gran", "e_Gran"
]

yu2018_full = pd.read_fwf("../0_inputdata_CDSxmatch/Yu2018_T1.dat", names=y18_columnnames)
yu2018_array = yu2018_full.to_records(index=False)

sub_array = pd.DataFrame({
    "KIC": yu2018_array["KIC"],
    "e_Delnu": yu2018_array["e_Delnu"],
    "e_numax": yu2018_array["e_numax"]
})

array_full = pd.merge(array_full, sub_array, on="KIC", how="inner")

Curate simplfied array.

In [12]:
y18 = pd.DataFrame({
    # star identifiers
    "gaiadr3_id": array_full["Source"],
    "ra": array_full["RAdeg"],
    "dec": array_full["DEdeg"],
    # asteroseismic parameters
    "numax": array_full["numax"],
    "e_numax": array_full["e_numax"],
    "dnu": array_full["Delnu"],
    "e_dnu": array_full["e_Delnu"],
    "ev_state": array_full["Phase"],
    # Gaia parameters
    "GAIA-teff": array_full["Teff.1"],
    "GAIA-e_teff": (array_full["B_Teff"] - array_full["b_Teff"]) / 2,
    "GAIA-logg": array_full["logg"],
    "GAIA-e_logg": (array_full["B_logg"] - array_full["b_logg"]) / 2,
    "GAIA-fe_h": array_full["[Fe/H]"],
    "GAIA-e_fe_h": (array_full["B_[Fe/H]"] - array_full["b_[Fe/H]"]) / 2,
    # datasets
    "h23": np.full(len(array_full), int(0)),
    "s26": np.full(len(array_full), int(0)),
    "w26": np.full(len(array_full), int(0)),
    "y18": np.full(len(array_full), int(1)),
    "z22": np.full(len(array_full), int(0)),
    # GSP Spec parameters
    "GSPSPEC-teff": array_full["Teff-S"],
    "GSPSPEC-e_teff": (array_full["B_Teff-S"] - array_full["b_Teff-S"]) / 2,
    "GSPSPEC-logg": array_full["logg-S"],
    "GSPSPEC-e_logg": (array_full["B_logg-S"] - array_full["b_logg-S"]) / 2,
    "GSPSPEC-fe_h": array_full["[Fe/H]-S"],
    "GSPSPEC-e_fe_h": (array_full["B_[Fe/H]-S"] - array_full["b_[Fe/H]-S"]) / 2,
    "GSPSPEC-si_fe": array_full["[Si/Fe]-S"],
    "GSPSPEC-e_si_fe": (array_full["B_[Si/Fe]-S"] - array_full["b_[Si/Fe]-S"]) / 2,
    "GSPSPEC-ca_fe": array_full["[Ca/Fe]-S"],
    "GSPSPEC-e_ca_fe": (array_full["B_[Ca/Fe]-S"] - array_full["b_[Ca/Fe]-S"]) / 2,
    "GSPSPEC-ti_fe": array_full["[Ti/Fe]-S"],
    "GSPSPEC-e_ti_fe": (array_full["B_[Ti/Fe]-S"] - array_full["b_[Ti/Fe]-S"]) / 2,
    "GSPSPEC-mg_fe": array_full["[Mg/Fe]-S"],
    "GSPSPEC-e_mg_fe": (array_full["B_[Mg/Fe]-S"] - array_full["b_[Mg/Fe]-S"]) / 2,
    "GSPSPEC-nd_fe": array_full["[Nd/Fe]-S"],
    "GSPSPEC-e_nd_fe": (array_full["B_[Nd/Fe]-S"] - array_full["b_[Nd/Fe]-S"]) / 2,
    "GSPSPEC-s_fe": array_full["[S/Fe]-S"],
    "GSPSPEC-e_s_fe": (array_full["B_[S/Fe]-S"] - array_full["b_[S/Fe]-S"]) / 2,
    "GSPSPEC-zr_fe": array_full["[Zr/Fe]-S"],
    "GSPSPEC-e_zr_fe": (array_full["B_[Zr/Fe]-S"] - array_full["b_[Zr/Fe]-S"]) / 2,
    "GSPSPEC-n_fe": array_full["[N/Fe]-S"],
    "GSPSPEC-e_n_fe": (array_full["B_[N/Fe]-S"] - array_full["b_[N/Fe]-S"]) / 2,
    "GSPSPEC-cr_fe": array_full["[Cr/Fe]-S"],
    "GSPSPEC-e_cr_fe": (array_full["B_[Cr/Fe]-S"] - array_full["b_[Cr/Fe]-S"]) / 2,
    "GSPSPEC-ce_fe": array_full["[Ce/Fe]-S"],
    "GSPSPEC-e_ce_fe": (array_full["B_[Ce/Fe]-S"] - array_full["b_[Ce/Fe]-S"]) / 2,
    "GSPSPEC-ni_fe": array_full["[Ni/Fe]-S"],
    "GSPSPEC-e_ni_fe": (array_full["B_[Ni/Fe]-S"] - array_full["b_[Ni/Fe]-S"]) / 2
})

y18 = y18.dropna(subset=["e_numax", "e_dnu"])

Taken within $0.75\times$ and $1.35\times$ of $\Delta\nu$-$\nu_\mathrm{max}$ relation (Stello+ 09, Mosser+ 11):

$$ \Delta\nu = 0.27 \nu_\mathrm{max}^{0.76}. $$

In [13]:
def relation(numax):
    coeff = 0.27
    return coeff * numax**(0.76)

mask = (y18["dnu"] < 1.35*relation(y18["numax"])) & (y18["dnu"] > 0.75*relation(y18["numax"]))

y18 = y18[mask]

Write to `.csv` file.

In [14]:
y18.to_csv("./yu18_asteroseismic.csv", index=False)